In [1]:
import pandas as pd
import math
import time

df = pd.read_csv("teen.csv")
# Remove ID column if present
#if "id" in df.clumns:
    #df = df.drop("id", axis=1)

target = df.columns[-1]
attributes = list(df.columns[:-1])

n = int(0.8 * len(df))

train = df.iloc[:n]
test = df.iloc[n:]

print("Target:", target)
print("Training rows:", len(train))
print("Testing rows:", len(test))

Target: depression_label
Training rows: 960
Testing rows: 240


In [2]:
def entropy(data):
    e = 0
    for count in data[target].value_counts():
        p = count / len(data)
        if p > 0:
            e -= p * math.log2(p)
    return e


def information_gain(data, attribute):
    weighted = 0
    for value in data[attribute].unique():
        subset = data[data[attribute] == value]

        weighted += (
            len(subset) / len(data)
        ) * entropy(subset)

    return entropy(data) - weighted


def gain_ratio(data, attribute):
    split_info = 0

    for value in data[attribute].unique():
        p = len(data[data[attribute] == value]) / len(data)

        if p > 0:
            split_info -= p * math.log2(p)

    if split_info == 0:
        return 0

    return information_gain(data, attribute) / split_info


def gini(data):
    g = 1

    for count in data[target].value_counts():
        p = count / len(data)
        g -= p ** 2

    return g

In [3]:
def build_tree(data, attributes, method):

    if data[target].nunique() == 1:
        return data[target].iloc[0]

    if not attributes: 
        return data[target].mode()[0]

    measure = information_gain if method == "ID3" else gain_ratio
    best = max(attributes, key=lambda x: measure(data, x))
    tree = {best: {}}

    remaining = [a for a in attributes if a != best]
    for value in data[best].unique():

        subset = data[data[best] == value]

        tree[best][value] = build_tree(
            subset, remaining, method
        )

    return tree

def id3(data, attributes):
    return build_tree(data, attributes, "ID3")

def c45(data, attributes):
    return build_tree(data, attributes, "C4.5")

In [4]:
def cart(data, attributes):

    if data[target].nunique() == 1:
        return data[target].iloc[0]

    if len(attributes) == 0:
        return data[target].mode()[0]

    best_score = float("inf")
    best_attribute = None
    best_value = None

    for attribute in attributes:
        for value in data[attribute].unique():

            left = data[data[attribute] == value]
            right = data[data[attribute] != value]

            if len(left) == 0 or len(right) == 0:
                continue

            score = (
                len(left) / len(data) * gini(left)
                +
                len(right) / len(data) * gini(right)
            )

            if score < best_score:
                best_score = score
                best_attribute = attribute
                best_value = value

    if best_attribute is None:
        return data[target].mode()[0]

    left = data[data[best_attribute] == best_value]
    right = data[data[best_attribute] != best_value]

    remaining = [
        a for a in attributes
        if a != best_attribute
    ]

    return {
        "attribute": best_attribute,
        "value": best_value,
        "left": cart(left, remaining),
        "right": cart(right, remaining)
    }

In [5]:
def predict(tree, row, method):

    if not isinstance(tree, dict):
        return tree

    if method == "CART":
        if row[tree["attribute"]] == tree["value"]:
            return predict(tree["left"], row, method)
        return predict(tree["right"], row, method)

    root = next(iter(tree))
    value = row[root]

    if value in tree[root]:
        return predict(tree[root][value], row, method)

    return train[target].mode()[0]


def depth(tree, method):

    if not isinstance(tree, dict):
        return 0

    if method == "CART":
        return 1 + max(depth(tree["left"], method),
                       depth(tree["right"], method))

    root = next(iter(tree))
    return 1 + max(depth(x, method)
                   for x in tree[root].values())


def leaves(tree, method):

    if not isinstance(tree, dict):
        return 1

    if method == "CART":
        return (leaves(tree["left"], method) +
                leaves(tree["right"], method))

    root = next(iter(tree))
    return sum(leaves(x, method)
               for x in tree[root].values())

In [6]:
def evaluate(actual, predicted):

    classes = df[target].unique()

    accuracy = sum(a == p for a, p in zip(actual, predicted)) / len(actual)

    precision = recall = f1 = 0

    for c in classes:

        tp = sum(a == c and p == c for a, p in zip(actual, predicted))
        fp = sum(a != c and p == c for a, p in zip(actual, predicted))
        fn = sum(a == c and p != c for a, p in zip(actual, predicted))

        p = tp / (tp + fp) if tp + fp else 0
        r = tp / (tp + fn) if tp + fn else 0

        precision += p
        recall += r
        f1 += 2 * p * r / (p + r) if p + r else 0

    n = len(classes)

    return accuracy, precision / n, recall / n, f1 / n

In [7]:
algorithms = {
    "ID3": id3,
    "C4.5": c45,
    "CART": cart
}

results = []

for name, algo in algorithms.items():

    start = time.perf_counter()
    tree = algo(train, attributes)
    t = time.perf_counter() - start

    actual = list(test[target])
    predicted = [predict(tree, row, name) for _, row in test.iterrows()]

    acc, pre, rec, f1 = evaluate(actual, predicted)

    results.append([
        name,
        acc * 100,
        pre,
        rec,
        f1,
        depth(tree, name),
        leaves(tree, name),
        t
    ])

result = pd.DataFrame(results, columns=[
    "Algorithm",
    "Accuracy (%)",
    "Precision",
    "Recall",
    "F1-Score",
    "Tree Depth",
    "Leaf Nodes",
    "Training Time"
])

display(result)

,Algorithm,Accuracy (%),Precision,Recall,F1-Score,Tree Depth,Leaf Nodes,Training Time
0,ID3,95.833333,0.483193,0.495690,0.489362,2,286,2.335519
1,C4.5,95.833333,0.483193,0.495690,0.489362,3,253,3.372481
2,CART,93.750000,0.482833,0.484914,0.483871,11,38,5.833586
